# Function Calling with IBM Granite 4

- <https://www.ibm.com/new/announcements/ibm-granite-4-0-hyper-efficient-high-performance-hybrid-models>
- <https://huggingface.co/ibm-granite/granite-4.0-h-micro>

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
device = "mps"
model_path = "ibm-granite/granite-4.0-h-micro"
# "ibm-granite/granite-4.0-h-small"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, device_map=device)

In [ ]:
tokenizer

## Chat Template

In [ ]:
print(tokenizer.get_chat_template())

In [ ]:
model.eval()
# change input text as desired
chat = [
    { "role": "user", "content": "How is the weather in London tomorrow?" },
]
chat = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
chat

In [ ]:
# tokenize the text
input_tokens = tokenizer(chat, return_tensors="pt").to(device)

In [ ]:
# generate output tokens
output = model.generate(**input_tokens, 
                        max_new_tokens=100)
# decode output tokens into text
output = tokenizer.batch_decode(output)
# print output
print(output[0])

## Function Calling

In [ ]:
tools = [
    {
        "name": "get_weather",
        "description": "Returns the weather for date and location",
        "parameters": {
            "type": "object",
            "properties": {
                "date": {"type": "string"},
                "location": {"type": "string"}
            },
            "required": ["date", "location"]
        }
    }
]

In [ ]:
# add tools to chat template
chat = [
    { "role": "user", "content": "How is the weather in London tomorrow?" },
]
chat = tokenizer.apply_chat_template(chat, tools=tools, tokenize=False, add_generation_prompt=True)

In [ ]:
print(chat)

In [ ]:
# tokenize the text
input_tokens = tokenizer(chat, return_tensors="pt").to(device)
# generate output tokens
output = model.generate(**input_tokens, 
                        max_new_tokens=100)
# decode output tokens into text
output = tokenizer.batch_decode(output)
# print output
print(output[0])

In [ ]:
tools = [{
  "name": "get_weather",
  "description": "Get weather forecast for a specific date and location.",
  "parameters": {
    "type": "object",
    "properties": {
      "location": {"type": "string"},
      "date": {
        "type": "string",
        "description": "An ISO date (YYYY-MM-DD). Convert relative dates like 'tomorrow' before filling this field."
      }
    },
    "required": ["location", "date"]
  }
}
]

In [ ]:
# change input text as desired
from datetime import datetime
current_date = datetime.now().strftime("%Y-%m-%d")
chat = [
    { "role": "system", "content": f'Assume todays date is {current_date}' },
    { "role": "user", "content": "How is the weather in London tomorrow?" },
]
chat = tokenizer.apply_chat_template(chat, tools=tools, tokenize=False, add_generation_prompt=True)

In [ ]:
print(chat)

In [ ]:
# tokenize the text
input_tokens = tokenizer(chat, return_tensors="pt").to(device)
# generate output tokens
output = model.generate(**input_tokens, 
                        max_new_tokens=100)
# decode output tokens into text
output = tokenizer.batch_decode(output)
# print output
print(output[0])